<a href="https://colab.research.google.com/github/warry258/colab/blob/main/GGUF%E8%9E%8D%E5%90%88LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title 💾 挂载Google硬盘
%%capture
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# @title ⚙️ 2. 初始化核心环境 (ComfyUI & GGUF)
import os, subprocess, sys, shutil

COMFY_ROOT = '/content/ComfyUI'
CUSTOM_NODES = f'{COMFY_ROOT}/custom_nodes'

print("📦 正在拉取基础组件库 (约需30秒)...")

# 1. 克隆最小化 ComfyUI 核心（仅作为解析引擎）
if not os.path.exists(COMFY_ROOT):
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/comfyanonymous/ComfyUI', COMFY_ROOT])

# 2. 克隆 ComfyUI-GGUF 插件
gguf_path = f'{CUSTOM_NODES}/ComfyUI-GGUF'
if not os.path.exists(gguf_path):
    subprocess.run(['git', 'clone', '--quiet', 'https://github.com/city96/ComfyUI-GGUF', gguf_path])

# 将插件重命名以支持 Python 包形式引入
gguf_new_path = f'{CUSTOM_NODES}/ComfyUI_GGUF'
if os.path.exists(gguf_path) and not os.path.exists(gguf_new_path):
    shutil.move(gguf_path, gguf_new_path)

# 3. 安装依赖（确保兼容 CPU）
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch', 'gguf', 'huggingface_hub', 'ipywidgets'])

# 4. 创建模型目录
DIR_UNET = os.path.join(COMFY_ROOT, "models", "unet")
DIR_LORA = os.path.join(COMFY_ROOT, "models", "loras")
os.makedirs(DIR_UNET, exist_ok=True)
os.makedirs(DIR_LORA, exist_ok=True)

import torch
device_type = "GPU 🚀" if torch.cuda.is_available() else "CPU 🐢"
print(f"✅ 环境配置完成！当前运行硬件: {device_type} (两者均支持防OOM融合)")

In [ ]:
# @title 📥 3. 极速下载底模与 LoRA (完整浏览版 + URL直链)
import os
import requests
import ipywidgets as widgets
from IPython.display import display
from huggingface_hub import hf_hub_download, list_repo_files

# 确保目录存在
DIR_UNET = "/content/ComfyUI/models/unet"
DIR_LORA = "/content/ComfyUI/models/loras"
os.makedirs(DIR_UNET, exist_ok=True)
os.makedirs(DIR_LORA, exist_ok=True)

out_dl = widgets.Output()

# ── 核心下载与扫描函数 ──
def fetch_repo_files(repo_id, exts):
    try:
        files = list_repo_files(repo_id)
        valid_files = sorted([f for f in files if any(f.endswith(e) for e in exts)])
        return valid_files if valid_files else["(未找到支持的文件格式)"]
    except Exception as e:
        return[f"(ERROR: 找不到仓库或网络错误)"]

def download_hf_file(repo_id, filename, dest_dir, type_str):
    if filename.startswith("("):
        with out_dl: print(f"❌ 无效的文件名，跳过下载。")
        return
    try:
        with out_dl:
            print(f"⏳ 正在从 HuggingFace 拉取 {type_str}: {filename} ...")
            downloaded = hf_hub_download(repo_id=repo_id, filename=filename, local_dir=dest_dir)
            print(f"✅ 下载成功: {downloaded}\n")
    except Exception as e:
        with out_dl: print(f"❌ 下载失败: {e}\n")

def download_url_file(url, filename, dest_dir):
    url = url.strip()
    if not url:
        with out_dl: print(f"❌ URL 不能为空。")
        return

    # 如果没填文件名，尝试从 URL 解析
    if not filename.strip():
        filename = url.split('?')[0].split('/')[-1]
        if not filename:
            filename = "downloaded_model.bin"

    target_path = os.path.join(dest_dir, filename.strip())

    try:
        with out_dl:
            print(f"⏳ 正在通过直链下载到 {dest_dir}: {filename} ...")
        r = requests.get(url, stream=True)
        r.raise_for_status()
        with open(target_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=8192):
                if chunk: f.write(chunk)
        with out_dl:
            print(f"✅ 直链下载成功: {target_path}\n")
    except Exception as e:
        with out_dl: print(f"❌ 直链下载失败: {e}\n")

# ── 样式定义 ──
S_BTN = widgets.Layout(width="100px", height="32px")
S_INPUT = widgets.Layout(flex="1 1 auto")

# ==========================================
# 1. GGUF 底模下载区 (HuggingFace)
# ==========================================
unet_preset = widgets.Dropdown(
    options=[
        ("📦 自定义", ""),
        ("unsloth/Z-Image-Turbo-GGUF", "unsloth/Z-Image-Turbo-GGUF"),
        ("unsloth/Z-Image-GGUF", "unsloth/Z-Image-GGUF"),
        ("unsloth/ERNIE-Image-Turbo-GGUF", "unsloth/ERNIE-Image-Turbo-GGUF"),
    ],
    value="unsloth/Z-Image-Turbo-GGUF",
    layout=widgets.Layout(width="220px")
)
unet_input = widgets.Text(value="unsloth/Z-Image-Turbo-GGUF", description="Repo:", layout=S_INPUT)
unet_browse_btn = widgets.Button(description="📂 浏览仓库", button_style="info", layout=S_BTN)
unet_file_dd = widgets.Dropdown(options=["(点击浏览加载文件)"], description="选择文件:", layout=widgets.Layout(width="100%"))
unet_dl_btn = widgets.Button(description="⬇️ 下载选中 GGUF", button_style="primary", layout=widgets.Layout(width="200px"))

def _on_unet_preset(c):
    if c['new']: unet_input.value = c['new']
unet_preset.observe(_on_unet_preset, names='value')

def _on_unet_browse(_):
    unet_file_dd.options = ["⏳ 获取文件列表中..."]
    files = fetch_repo_files(unet_input.value.strip(), ('.gguf',))
    unet_file_dd.options = files
unet_browse_btn.on_click(_on_unet_browse)
unet_dl_btn.on_click(lambda _: download_hf_file(unet_input.value.strip(), unet_file_dd.value, DIR_UNET, "GGUF底模"))

# ==========================================
# 2. LoRA 下载区 (HuggingFace)
# ==========================================
lora_preset = widgets.Dropdown(
    options=[
        ("📦 自定义", ""),
        ("alibaba-pai/Z-Image-Fun-Lora-Distill", "alibaba-pai/Z-Image-Fun-Lora-Distill"),
        ("nphSi/Z-Image-Lora", "nphSi/Z-Image-Lora"),
        ("ifmylove2011/girlslike-zimage", "ifmylove2011/girlslike-zimage"),
    ],
    value="alibaba-pai/Z-Image-Fun-Lora-Distill",
    layout=widgets.Layout(width="220px")
)
lora_input = widgets.Text(value="alibaba-pai/Z-Image-Fun-Lora-Distill", description="Repo:", layout=S_INPUT)
lora_browse_btn = widgets.Button(description="📂 浏览仓库", button_style="info", layout=S_BTN)
lora_file_dd = widgets.Dropdown(options=["(点击浏览加载文件)"], description="选择文件:", layout=widgets.Layout(width="100%"))
lora_dl_btn = widgets.Button(description="⬇️ 下载选中 LoRA", button_style="success", layout=widgets.Layout(width="200px"))

def _on_lora_preset(c):
    if c['new']: lora_input.value = c['new']
lora_preset.observe(_on_lora_preset, names='value')

def _on_lora_browse(_):
    lora_file_dd.options =["⏳ 获取文件列表中..."]
    files = fetch_repo_files(lora_input.value.strip(), ('.safetensors', '.pt'))
    lora_file_dd.options = files
lora_browse_btn.on_click(_on_lora_browse)
lora_dl_btn.on_click(lambda _: download_hf_file(lora_input.value.strip(), lora_file_dd.value, DIR_LORA, "LoRA模型"))

# ==========================================
# 3. 直链 URL 下载区 (Civitai等)
# ==========================================
url_input = widgets.Text(placeholder="输入直链地址，如: https://.../model.safetensors", description="URL直链:", layout=widgets.Layout(flex="1 1 auto"))
url_filename = widgets.Text(placeholder="可留空(自动解析)", description="保存名称:", layout=widgets.Layout(width="250px"))
url_type_dd = widgets.Dropdown(
    options=[("🟦 GGUF底模 (/models/unet)", DIR_UNET), ("🟩 LoRA (/models/loras)", DIR_LORA)],
    description="目标路径:",
    layout=widgets.Layout(width="300px")
)
url_dl_btn = widgets.Button(description="🔗 开始 URL 下载", button_style="warning", layout=widgets.Layout(width="200px"))

url_dl_btn.on_click(lambda _: download_url_file(url_input.value, url_filename.value, url_type_dd.value))

# ==========================================
# UI 布局组合
# ==========================================
html_title = widgets.HTML("""
<div style='background:#0f172a; color:white; padding:15px; border-radius:10px; border:1px solid #334155;'>
  <h3 style='margin:0 0 5px 0; color:#38bdf8;'>📦 模型仓库浏览与多通道下载</h3>
  <span style='font-size:12px; color:#94a3b8;'>支持 HuggingFace 仓库极速抓取，同时也支持 Civitai 等外部网站的直链下载。</span>
</div>
""")

ui = widgets.VBox([
    html_title,

    # HF GGUF
    widgets.HTML("<hr style='border-color:#334155;'><b>🟦 HuggingFace 下载: 基础 GGUF 模型</b>"),
    widgets.HBox([unet_preset, unet_input, unet_browse_btn]),
    unet_file_dd,
    unet_dl_btn,

    # HF LoRA
    widgets.HTML("<hr style='border-color:#334155;'><b>🟩 HuggingFace 下载: LoRA 模型</b>"),
    widgets.HBox([lora_preset, lora_input, lora_browse_btn]),
    lora_file_dd,
    lora_dl_btn,

    # URL 下载
    widgets.HTML("<hr style='border-color:#334155;'><b>🟨 外部直链下载 (URL)</b>"),
    widgets.HBox([url_input, url_filename]),
    widgets.HBox([url_type_dd]),
    url_dl_btn,

    # 输出终端
    widgets.HTML("<hr style='border-color:#334155;'>"),
    out_dl
], layout=widgets.Layout(padding="20px", border="1px solid #1e293b", border_radius="10px"))

display(ui)

In [ ]:
# @title 🍳 4. Zero-RAM 离线融合器 (物理覆写防爆版)
import os, sys, shutil, gc, time, logging, contextlib
import torch
import ipywidgets as widgets
from IPython.display import display, clear_output

# ========== 【关键修复 1】补全正确的路径环境与工作目录 ==========
COMFY_ROOT = '/content/ComfyUI'
CUSTOM_NODES = f'{COMFY_ROOT}/custom_nodes'

if COMFY_ROOT not in sys.path:
    sys.path.insert(0, COMFY_ROOT)
if CUSTOM_NODES not in sys.path:
    sys.path.insert(0, CUSTOM_NODES) # 必须让 Python 知道去哪找 ComfyUI_GGUF

# 强制切换工作目录到 ComfyUI 根目录，防止某些组件报错
if os.path.exists(COMFY_ROOT):
    os.chdir(COMFY_ROOT)
# ================================================================

# ========== 【关键修复 2】彻底阻断 ComfyUI 的强制 GPU 检查 ==========
if 'comfy.cli_args' in sys.modules:
    if not torch.cuda.is_available():
        sys.modules['comfy.cli_args'].args.cpu = True

_orig_argv = sys.argv
sys.argv = [sys.argv[0], '--cpu'] if not torch.cuda.is_available() else [sys.argv[0]]

try:
    import comfy.cli_args
    if not torch.cuda.is_available():
        comfy.cli_args.args.cpu = True  # 双重保险，物理锁死 CPU 模式

    import comfy.utils
    import comfy.sd
finally:
    sys.argv = _orig_argv  # 恢复环境
# ====================================================================

import gguf

# ── 拦截扰民警告 ──
@contextlib.contextmanager
def suppress_lora_warnings():
    logger = logging.getLogger()
    class LoraWarningFilter(logging.Filter):
        def filter(self, record):
            return "lora key not loaded" not in record.getMessage()
    f = LoraWarningFilter()
    logger.addFilter(f)
    try: yield
    finally: logger.removeFilter(f)

# ── 魔法核心：拦截 GGUF 强制开启 r+ 写权限（内存零损耗关键） ──
if not hasattr(gguf.GGUFReader, "_original_init_for_bake"):
    gguf.GGUFReader._original_init_for_bake = gguf.GGUFReader.__init__

def _rplus_init(self, path, mode='r'):
    gguf.GGUFReader._original_init_for_bake(self, path, mode='r+')

def get_files(path, exts):
    if not os.path.exists(path): return []
    return sorted([f for f in os.listdir(path) if f.endswith(exts)])

# ── 构建 UI ──
DIR_UNET = "/content/ComfyUI/models/unet"
DIR_LORA = "/content/ComfyUI/models/loras"

out_bake = widgets.Output()
dd_unet = widgets.Dropdown(options=get_files(DIR_UNET, ('.gguf',)), description='基础GGUF:', layout=widgets.Layout(width='80%'))
text_out = widgets.Text(description='输出命名:', placeholder='例如: baked_mix.gguf', layout=widgets.Layout(width='80%'))
btn_bake = widgets.Button(description='🔥 串联开始烘焙', button_style='danger', layout=widgets.Layout(width='200px', height='40px'))

lora_slots =[]
slots_container = widgets.VBox([])

def add_slot(_=None):
    opts =[("（不加载）", "")] +[(f, f) for f in get_files(DIR_LORA, ('.safetensors', '.pt'))]
    dd = widgets.Dropdown(options=opts, value="", description=f"LoRA槽{len(lora_slots)+1}:", layout=widgets.Layout(width='50%'))
    sm = widgets.FloatSlider(value=1.0, min=-2.0, max=3.0, step=0.05, description='强度:', layout=widgets.Layout(width='30%'))
    lora_slots.append((dd, sm))
    slots_container.children = list(slots_container.children) + [widgets.HBox([dd, sm])]

add_slot() # 默认1个槽
btn_add_slot = widgets.Button(description='➕ 添加 LoRA', button_style='info')
btn_add_slot.on_click(add_slot)

# ── 核心引擎 ──
def execute_bake(_):
    out_bake.clear_output()
    btn_bake.disabled = True
    try:
        with out_bake:
            base_unet = dd_unet.value
            out_name  = text_out.value.strip() or f"Baked_{base_unet}"
            if not out_name.endswith('.gguf'): out_name += '.gguf'

            active_loras =[(dd.value, sm.value) for dd, sm in lora_slots if dd.value and abs(sm.value) > 0.001]
            if not base_unet or not active_loras:
                print("❌ 错误：请选择底模和至少一个有效的 LoRA。")
                return

            base_path = os.path.join(DIR_UNET, base_unet)
            out_path  = os.path.join(DIR_UNET, out_name)

            print("="*60)
            print(f"🚀 开始防OOM融合 (Zero-RAM模式)")
            print(f"   输入: {base_unet} \n   输出: {out_name}")
            print("="*60)

            print("⏳ 1/4 正在克隆底模至输出路径 (物理文件拷贝)...")
            shutil.copy2(base_path, out_path)

            print("⏳ 2/4 挂载 mmap(r+) 读写权限并拉起解析器...")
            gguf.GGUFReader.__init__ = _rplus_init

            # 【这里就不会报 ModuleNotFoundError 了】
            from ComfyUI_GGUF import nodes as gguf_nodes
            loader = gguf_nodes.UnetLoaderGGUF()
            (model_patcher,) = loader.load_unet(out_name)

            print(f"⏳ 3/4 链式解析 {len(active_loras)} 个 LoRA 结构...")
            with suppress_lora_warnings():
                for lname, lstr in active_loras:
                    lpath = os.path.join(DIR_LORA, lname)
                    print(f"   🔗 正在提取并叠加账本: {lname} (强度: {lstr}) ...")
                    lora_sd = comfy.utils.load_torch_file(lpath)
                    model_patcher, _ = comfy.sd.load_lora_for_models(model_patcher, None, lora_sd, lstr, 0.0)
                    del lora_sd
                    gc.collect() # 极致垃圾回收

            print("⏳ 4/4 正在逐层进行磁盘物理覆写 (防爆内存运作中)...")
            patches = model_patcher.patches
            total_layers = len(patches)
            count_success = count_skipped = 0

            for i, (key, layer_patches) in enumerate(patches.items()):
                if not layer_patches: continue
                raw_param = model_patcher.model.state_dict().get(key)
                if raw_param is None: continue
                raw_tensor = raw_param.data if hasattr(raw_param, 'data') else raw_param

                # GGUF 量化层不支持直接合并，只合并 F16/F32 层
                if raw_tensor.dtype not in[torch.float16, torch.float32]:
                    count_skipped += 1
                    continue

                try:
                    new_weight = model_patcher.calculate_weight(layer_patches, raw_tensor, key)
                    # ★ 核心动作：直接写入物理内存映射，不需要申请多余显存/内存
                    raw_tensor.copy_(new_weight.to(raw_tensor.dtype).cpu())

                    del new_weight
                    gc.collect()
                    count_success += 1
                    if count_success % 20 == 0: print(f"   🔨 已覆写 {count_success}/{total_layers} 个张量...")
                except Exception as e:
                    print(f"   ⚠️ 张量 {key} 融合失败: {e}")

            print(f"\n🎉 烘焙大功告成！成功修改 {count_success} 个张量，保护跳过 {count_skipped} 个非浮点层。")
            print("="*60)

    except Exception as e:
        with out_bake:
            print(f"\n❌ 致命异常: {e}")
            import traceback; traceback.print_exc()
    finally:
        gguf.GGUFReader.__init__ = gguf.GGUFReader._original_init_for_bake
        if 'model_patcher' in locals(): del model_patcher
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        btn_bake.disabled = False

btn_bake.on_click(execute_bake)

display(widgets.VBox([
    widgets.HTML("""
    <div style="background:#1e1b4b; padding:15px; border-radius:10px; border:1px solid #4338ca; margin-bottom:10px;">
        <h3 style="color:#a5b4fc; margin:0 0 5px 0;">🍳 GGUF-LoRA Zero-RAM 离线融合器 </h3>
        <span style="color:#818cf8; font-size:12px;">支持无限追加 LoRA，纯 CPU 环境防爆优化版，物理级别直接修改文件。</span>
    </div>
    """),
    dd_unet, slots_container, btn_add_slot, text_out, btn_bake, out_bake
], layout=widgets.Layout(padding="20px", border="1px solid #475569", border_radius="10px", background="#0f172a")))

In [ ]:
# @title 保存至 Google Drive
import shutil
import os
import datetime
import ipywidgets as widgets
from IPython.display import display

def get_baked_files():
    if not os.path.exists(DIR_UNET): return []
    return sorted([f for f in os.listdir(DIR_UNET) if f.endswith('.gguf') and f != dd_unet.value])

# --- UI组件定义 ---
export_dd = widgets.Dropdown(options=get_files(DIR_UNET, ('.gguf',)), description='选择成品:', layout=widgets.Layout(width='80%'))

folder_name_input = widgets.Text(
    value='/content/drive/MyDrive/models',  # 默认填充Google Drive常用路径
    placeholder='输入完整路径(如/content/drive/MyDrive/models)或文件夹名',
    description='目标路径:',
    layout=widgets.Layout(width='80%')
)

btn_export = widgets.Button(description='📤 移至 Google Drive', button_style='success')
out_export = widgets.Output()

def export_to_drive(_):
    out_export.clear_output()
    with out_export:
        target_file = export_dd.value
        if not target_file:
            print("❌ 没有选择文件。")
            return

        user_input = folder_name_input.value.strip()

        # 1. 确定最终目标目录
        if user_input.startswith('/'):
            # 用户输入了绝对路径，直接使用
            base_dir = user_input
            safe_folder_name = ''
        else:
            # 用户输入了文件夹名，使用默认基础路径
            base_dir = '/content/drive/MyDrive/models'  # 这里硬编码默认路径，解决SAVE_DIR未定义问题
            if user_input:
                # 清洗文件夹名
                safe_folder_name = "".join([c for c in user_input if c.isalnum() or c in (' ', '_', '-')]).rstrip()
                if not safe_folder_name:
                    print("⚠️ 文件夹名称无效，已自动切换为时间戳模式。")
                    safe_folder_name = f"gguf_export_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
            else:
                # 留空，使用时间戳
                safe_folder_name = f"gguf_export_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"

        # 2. 拼接最终路径
        src_path = os.path.join(DIR_UNET, target_file)

        if safe_folder_name:
            dst_dir = os.path.join(base_dir, safe_folder_name)
        else:
            dst_dir = base_dir

        dst_path = os.path.join(dst_dir, target_file)

        # 3. 创建文件夹并复制
        os.makedirs(dst_dir, exist_ok=True)

        print(f"⏳ 正在复制 {target_file} ...")
        print(f"📂 源文件: {src_path}")
        print(f"📂 目标文件: {dst_path}")
        print("(文件较大，可能需要1-3分钟，请耐心等待...)")

        shutil.copy2(src_path, dst_path)
        print(f"\n✅ 复制成功！")
        print(f"📌 文件已保存至: {dst_path}")

btn_export.on_click(export_to_drive)

# --- 显示界面 ---
display(widgets.VBox([
    widgets.HTML("选择刚刚生成好的模型，保存至 Drive 以备以后直接流式加载。"),
    export_dd,
    folder_name_input,
    btn_export,
    out_export
]))